In [ ]:

%pip install -q mediapipe==0.10.35 huggingface_hub scikit-learn seaborn


In [ ]:

import os
import json
import time
import random
import shutil
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image
from collections import Counter

from huggingface_hub import snapshot_download

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python environment ready")
print("MediaPipe:", mp.__version__)
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)

if not gpus:
    raise RuntimeError("No GPU detected. Select T4 GPU in Colab.")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPU training available!")


In [ ]:

REPO_ID = "ntsrigaud/hagrid-subset"

TARGET_CLASSES = [
    "palm",
    "fist",
    "like",
    "peace",
    "point",
    "ok"
]

CLASS_TO_NAME = {
    "palm": "Open Palm",
    "fist": "Fist",
    "like": "Thumbs Up",
    "peace": "Peace Sign",
    "point": "Pointing Finger",
    "ok": "OK Sign"
}

IMG_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 10

PROJECT_DIR = Path("/content/gesture_project")
PROCESSED_DIR = PROJECT_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

PROJECT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

CACHE_ROOT = (
    Path.home()
    / ".cache/huggingface/hub/datasets--ntsrigaud--hagrid-subset/snapshots"
)

def valid_snapshot(snapshot):
    for split in ["train", "val", "test"]:
        for gesture in TARGET_CLASSES:
            folder = snapshot / "data" / split / gesture
            if not folder.exists() or len(list(folder.glob("*.jpg"))) == 0:
                return False
    return True

snapshots = list(CACHE_ROOT.glob("*")) if CACHE_ROOT.exists() else []

DATA_DIR = next(
    (p for p in snapshots if valid_snapshot(p)),
    None
)

if DATA_DIR is None:
    print("Complete dataset cache not found. Downloading required subset...")

    patterns = ["annotations.csv"]

    for split in ["train", "val", "test"]:
        for gesture in TARGET_CLASSES:
            patterns.append(
                f"data/{split}/{gesture}/*.jpg"
            )

    DATA_DIR = Path(snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        allow_patterns=patterns
    ))

else:
    print("Reusing existing dataset cache.")

DATA_DIR = Path(DATA_DIR)

print("Dataset:", DATA_DIR)
print("Project:", PROJECT_DIR)
print("Dataset available:", valid_snapshot(DATA_DIR))


In [ ]:

counts = {}

for split in ["train", "val", "test"]:

    counts[split] = {}

    for gesture in TARGET_CLASSES:

        folder = DATA_DIR / "data" / split / gesture

        count = len(list(folder.glob("*.jpg")))

        counts[split][gesture] = count

        print(f"{split:5s} | {gesture:8s} | {count}")

    print("TOTAL:", sum(counts[split].values()))
    print("-" * 40)

annotations = pd.read_csv(DATA_DIR / "annotations.csv")

print("Annotation shape:", annotations.shape)
print("Annotation columns:", annotations.columns.tolist())

filtered_annotations = annotations[
    annotations["label"].isin(TARGET_CLASSES)
]

print("Filtered annotation count:", len(filtered_annotations))

assert sum(counts["train"].values()) == 3648
assert sum(counts["val"].values()) == 432
assert sum(counts["test"].values()) == 720

print("Dataset verification passed.")


In [ ]:

MODEL_PATH = "/content/hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    import urllib.request

    model_url = (
        "https://storage.googleapis.com/mediapipe-models/"
        "hand_landmarker/hand_landmarker/float16/1/"
        "hand_landmarker.task"
    )

    urllib.request.urlretrieve(model_url, MODEL_PATH)

print("Hand model available:", os.path.getsize(MODEL_PATH))

def create_detector():
    options = vision.HandLandmarkerOptions(
        base_options=python.BaseOptions(
            model_asset_path=MODEL_PATH
        ),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.2
    )

    return vision.HandLandmarker.create_from_options(options)


def square_hand_crop(image, landmarks, padding=0.35):

    h, w = image.shape[:2]

    xs = np.array([p.x * w for p in landmarks])
    ys = np.array([p.y * h for p in landmarks])

    x_center = (xs.min() + xs.max()) / 2
    y_center = (ys.min() + ys.max()) / 2

    box_size = max(
        xs.max() - xs.min(),
        ys.max() - ys.min()
    ) * (1 + 2 * padding)

    box_size = max(2, int(box_size))

    half = box_size // 2

    x1 = int(x_center) - half
    y1 = int(y_center) - half
    x2 = x1 + box_size
    y2 = y1 + box_size

    # Preserve hand regions near image edges with padding
    left = max(0, -x1)
    top = max(0, -y1)
    right = max(0, x2 - w)
    bottom = max(0, y2 - h)

    x1c = max(0, x1)
    y1c = max(0, y1)
    x2c = min(w, x2)
    y2c = min(h, y2)

    crop = image[y1c:y2c, x1c:x2c]

    if crop.size == 0:
        return None

    crop = cv2.copyMakeBorder(
        crop,
        top, bottom, left, right,
        cv2.BORDER_CONSTANT,
        value=(0, 0, 0)
    )

    crop = cv2.resize(
        crop,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_AREA
    )

    return crop


def detect_and_crop(image, detector):

    attempts = [
        image
    ]

    enlarged = cv2.resize(
        image,
        None,
        fx=1.5,
        fy=1.5,
        interpolation=cv2.INTER_CUBIC
    )

    attempts.append(enlarged)

    lab = cv2.cvtColor(enlarged, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced = cv2.cvtColor(
        cv2.merge((clahe.apply(l), a, b)),
        cv2.COLOR_LAB2RGB
    )

    attempts.append(enhanced)

    for candidate in attempts:

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=candidate
        )

        result = detector.detect(mp_image)

        if result.hand_landmarks:

            hand = max(
                result.hand_landmarks,
                key=lambda points: (
                    max(p.x for p in points) - min(p.x for p in points)
                ) * (
                    max(p.y for p in points) - min(p.y for p in points)
                )
            )

            # Normalized coordinates are valid for the original
            # image even if detection used an enlarged copy.
            crop = square_hand_crop(image, hand)

            if crop is not None:
                return crop

    return None

print("MediaPipe preprocessing functions ready.")


In [ ]:

fig, axes = plt.subplots(2, 6, figsize=(18, 7))

with create_detector() as detector:

    for i, gesture in enumerate(TARGET_CLASSES):

        folder = DATA_DIR / "data" / "train" / gesture

        image_path = random.choice(list(folder.glob("*.jpg")))

        image = np.array(
            Image.open(image_path).convert("RGB")
        )

        crop = detect_and_crop(image, detector)

        axes[0, i].imshow(image)
        axes[0, i].set_title(CLASS_TO_NAME[gesture])
        axes[0, i].axis("off")

        if crop is not None:
            axes[1, i].imshow(crop)
            axes[1, i].set_title("Detected crop")
        else:
            axes[1, i].text(
                0.5, 0.5, "Detection failed",
                ha="center", va="center"
            )

        axes[1, i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

from tqdm.auto import tqdm

SPLITS = ["train", "val", "test"]

for split in SPLITS:
    for gesture in TARGET_CLASSES:
        (PROCESSED_DIR / split / gesture).mkdir(
            parents=True,
            exist_ok=True
        )

audit = []

with create_detector() as detector:

    for split in SPLITS:

        for gesture in TARGET_CLASSES:

            source_folder = DATA_DIR / "data" / split / gesture

            image_paths = sorted(source_folder.glob("*.jpg"))

            success_count = 0
            failed_count = 0

            for image_path in tqdm(
                image_paths,
                desc=f"{split}/{gesture}"
            ):

                try:
                    image = np.array(
                        Image.open(image_path).convert("RGB")
                    )

                    crop = detect_and_crop(image, detector)

                    if crop is None:
                        failed_count += 1
                        audit.append({
                            "split": split,
                            "gesture": gesture,
                            "image": image_path.name,
                            "status": "failed"
                        })
                        continue

                    output_path = (
                        PROCESSED_DIR / split / gesture /
                        (image_path.stem + ".jpg")
                    )

                    crop_bgr = cv2.cvtColor(
                        crop,
                        cv2.COLOR_RGB2BGR
                    )

                    cv2.imwrite(
                        str(output_path),
                        crop_bgr,
                        [cv2.IMWRITE_JPEG_QUALITY, 95]
                    )

                    success_count += 1

                    audit.append({
                        "split": split,
                        "gesture": gesture,
                        "image": image_path.name,
                        "status": "success"
                    })

                except Exception as error:

                    failed_count += 1

                    audit.append({
                        "split": split,
                        "gesture": gesture,
                        "image": image_path.name,
                        "status": "error: " + str(error)
                    })

            print(
                f"{split}/{gesture}: "
                f"{success_count} successful, "
                f"{failed_count} failed"
            )

audit_df = pd.DataFrame(audit)

audit_df.to_csv(
    RESULTS_DIR / "detection_audit.csv",
    index=False
)

print("\nDetection audit completed.")

summary = pd.crosstab(
    audit_df["split"],
    audit_df["status"]
)

display(summary)

print("\nPer-class detection success:")
display(
    audit_df.pivot_table(
        index="gesture",
        columns="split",
        values="status",
        aggfunc=lambda x: (x == "success").sum()
    )
)


In [ ]:

AUTOTUNE = tf.data.AUTOTUNE

def create_dataset(split, shuffle):

    folder = str(PROCESSED_DIR / split)

    dataset = tf.keras.utils.image_dataset_from_directory(
        folder,
        class_names=TARGET_CLASSES,
        label_mode="int",
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED
    )

    return dataset.prefetch(AUTOTUNE)


train_ds = create_dataset("train", True)
val_ds = create_dataset("val", False)
test_ds = create_dataset("test", False)

print("Class order:", TARGET_CLASSES)

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)


In [ ]:

from tensorflow.keras import layers, Model

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10)
], name="data_augmentation")

# Pretrained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

# CNN architecture
inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.30)(x)

outputs = layers.Dense(
    len(TARGET_CLASSES),
    activation="softmax"
)(x)

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="HandGesture_MobileNetV2"
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

checkpoint_path = str(
    RESULTS_DIR / "best_gesture_model.keras"
)

callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

training_time = time.time() - start_time

print(
    f"Training time: {training_time / 60:.2f} minutes"
)


In [ ]:

# Load best validation checkpoint
model = tf.keras.models.load_model(checkpoint_path)

y_true = np.concatenate([
    labels.numpy()
    for _, labels in test_ds
])

probabilities = model.predict(test_ds)

y_pred = np.argmax(probabilities, axis=1)

# Accuracy
test_accuracy = accuracy_score(y_true, y_pred)

print("TEST ACCURACY:", round(test_accuracy * 100, 2), "%")

# Classification report
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(TARGET_CLASSES))),
    target_names=[
        CLASS_TO_NAME[c] for c in TARGET_CLASSES
    ],
    zero_division=0
)

print("\nCLASSIFICATION REPORT\n")
print(report)

with open(RESULTS_DIR / "classification_report.txt", "w") as f:
    f.write(report)

# Confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(TARGET_CLASSES)))
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[CLASS_TO_NAME[c] for c in TARGET_CLASSES],
    yticklabels=[CLASS_TO_NAME[c] for c in TARGET_CLASSES]
)

plt.xlabel("Predicted Gesture")
plt.ylabel("Actual Gesture")
plt.title("CNN Gesture Recognition - Confusion Matrix")
plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "confusion_matrix.png",
    dpi=200
)

plt.show()

# Training history
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Model Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "training_history.png",
    dpi=200
)

plt.show()

# Save numerical metrics
metrics = {
    "test_accuracy": float(test_accuracy),
    "training_time_seconds": float(training_time),
    "epochs_completed": len(history.history["loss"]),
    "training_images": int(sum(
        len(list((PROCESSED_DIR / "train" / c).glob("*.jpg")))
        for c in TARGET_CLASSES
    )),
    "validation_images": int(sum(
        len(list((PROCESSED_DIR / "val" / c).glob("*.jpg")))
        for c in TARGET_CLASSES
    )),
    "test_images": int(len(y_true))
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("\nSaved evaluation results.")


In [ ]:

from google.colab import files

EXPORT_DIR = PROJECT_DIR / "export"
EXPORT_DIR.mkdir(exist_ok=True)

# Save final CNN
MODEL_EXPORT = EXPORT_DIR / "gesture_cnn.keras"

model.save(MODEL_EXPORT)

# Save class labels in prediction order
with open(EXPORT_DIR / "class_names.json", "w") as f:
    json.dump(
        {
            str(i): {
                "label": gesture,
                "display_name": CLASS_TO_NAME[gesture]
            }
            for i, gesture in enumerate(TARGET_CLASSES)
        },
        f,
        indent=4
    )

# Copy evaluation evidence
for filename in [
    "confusion_matrix.png",
    "training_history.png",
    "classification_report.txt",
    "metrics.json",
    "detection_audit.csv"
]:

    source = RESULTS_DIR / filename

    if source.exists():
        shutil.copy2(source, EXPORT_DIR / filename)

# Package export
import zipfile

zip_path = "/content/gesture_cnn_export.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:

    for path in EXPORT_DIR.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                path.relative_to(EXPORT_DIR)
            )

print("MODEL:", MODEL_EXPORT)
print("MODEL SIZE:", round(MODEL_EXPORT.stat().st_size / 1024**2, 2), "MB")
print("EXPORT:", zip_path)

files.download(zip_path)
